# Immo Eliza - Data Overview
Just a quick look at our Immovlan dataset so we all know what we're working with.

If you're running this with a different or newer dataset, just change the file name in the cell below.

In [34]:
import pandas as pd

# change this path if you got a newer version of the dataset
DATASET_PATH = 'dataframe_info.json'

# loading the whole thing into a dataframe
df = pd.read_json(DATASET_PATH)

print(f"Dataset: {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

Dataset: 16729 rows, 21 columns
Columns: ['property_id', 'province', 'address', 'postcode', 'city', 'price', 'latitude', 'longitude', 'property_type', 'livable_surface', 'total_surface', 'bedroom_count', 'build_year', 'property_state', 'peb_category', 'garage', 'terrace', 'swimming_pool', 'Preschool_distance', 'Train_station_distance', 'Supermarket_distance']


## Missing Values
We already use NaN for empty stuff which is what we're supposed to do.
But some columns have a LOT of missing data so worth checking with the team what we wanna do about those.

In [35]:
# counting how many NaN per column and what % of the total that is
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(1)
})

# only showing columns that actually have missing values, worst first
missing = missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)
missing

,missing_count,missing_pct
swimming_pool,13444,80.4
garage,9030,54.0
total_surface,7314,43.7
build_year,6210,37.1
property_state,3812,22.8
peb_category,3326,19.9
terrace,3020,18.1
livable_surface,974,5.8
Train_station_distance,923,5.5
bedroom_count,400,2.4


## Duplicates Check
Just making sure we don't have the same property twice in the dataset.

In [36]:
# checking for full row duplicates and also just by property_id
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate property_id: {df.duplicated(subset=['property_id']).sum()}")

Duplicate rows: 0
Duplicate property_id: 0


## Blank Spaces Check
Looking for sneaky spaces before or after text values, like `" Brussels "` instead of `"Brussels"`.

In [37]:
# going through every text column and checking if any value has spaces around it
for col in df.select_dtypes(include='str').columns:
    bad = df[col].dropna().apply(lambda x: x != x.strip()).sum()
    print(f"{col}: {bad} values with leading/trailing spaces")

property_id: 0 values with leading/trailing spaces
province: 0 values with leading/trailing spaces
address: 0 values with leading/trailing spaces
city: 0 values with leading/trailing spaces
property_type: 0 values with leading/trailing spaces
property_state: 0 values with leading/trailing spaces


## Weird stuff to double check ?

### 1. garage, terrace, swimming_pool never say yes
These columns only have `0` (no) and `NaN` (unknown) but never `1` (yes).
So either the scraper didn't pick up the "yes" cases, or NaN actually means yes here.
Worth checking with whoever worked on the scraper.

In [38]:
# showing what values each binary column actually has
# we'd expect to see 0, 1, and NaN but we only see 0 and NaN
for col in ['garage', 'terrace', 'swimming_pool']:
    vals = df[col].value_counts(dropna=False).to_dict()
    print(f"{col}: {vals}")

garage: {nan: 9030, 0.0: 7699}
terrace: {0.0: 13709, nan: 3020}
swimming_pool: {nan: 13444, 0.0: 3285}


### 2. peb_category is actually energy consumption, not the letter grade
On immovlan this field comes from "Specific primary energy consumption" (kWh/m²/year),
not the PEB letter grade (A, B, C, D, E, F, G). So the column name is misleading.
Normal values sit roughly between 0 and 800 kWh/m²/year.
We got some negative ones and the max is like 20 million which looks like a date got stuck in there (20260607 = 2026-06-07).

In [39]:
# taking only the non-empty PEB values
# NOTE: these are kWh/m²/year from immovlan's "Specific primary energy consumption", not PEB letter grades
peb = df['peb_category'].dropna()

print(f"Energy consumption range: {peb.min()} to {peb.max()} kWh/m²/year")
print(f"Negative values: {(peb < 0).sum()}")
print(f"Over 1000 kWh/m²/year (looks off): {(peb > 1000).sum()}")
print()

# showing the really out there ones
print("The really big ones (probably dates or something else):")
print(df.loc[df['peb_category'] > 10000, ['property_id', 'peb_category', 'city']])
#1 hectare = 10000 m²

Energy consumption range: -52.0 to 20260607.0 kWh/m²/year
Negative values: 11
Over 1000 kWh/m²/year (looks off): 234

The really big ones (probably dates or something else):
      property_id  peb_category              city
292      RBV83965     3067345.0        Borgerhout
2553     RBW17995    20260607.0             Kermt
7248     VBE32901       51133.0            Spiere
7514     VBE32879       36701.0      Blankenberge
7700     RBV85959       11567.0           Waregem
...           ...           ...               ...
14134    VBE25685      112605.0          Bouillon
14327    VBE26262       14155.0             Namur
15248    VBE35057       27198.0            Tubize
16324    VBE11305       38821.0  Louvain-la-Neuve
16614    RBW23838      107971.0      Grez-Doiceau

[65 rows x 3 columns]


### 3. livable and total surface outliers
Got a few properties with like 1 m² livable surface which doesn't really make sense,
and one with a total surface of 178K m² which is like 17 hectares.
Could be real estates or just scraping weirdness.

In [40]:
# checking livable surface for stuff that looks too small or too big
ls = df['livable_surface'].dropna()
print(f"Livable surface range: {ls.min()} to {ls.max()} m²")
print(f"Under 10 m²: {(ls < 10).sum()}")
print(f"Over 1000 m²: {(ls > 1000).sum()}")
print()

# same for total surface
ts = df['total_surface'].dropna()
print(f"Total surface range: {ts.min()} to {ts.max()} m²")
print(f"That max is {ts.max()/10000:.1f} hectares")

Livable surface range: 1.0 to 5000.0 m²
Under 10 m²: 11
Over 1000 m²: 63

Total surface range: 1.0 to 178367.0 m²
That max is 17.8 hectares


### 4. build year goes into the future
Some properties have a build year like 2027 or 2028. Could be new construction that's planned, not necessarily wrong.

In [41]:
# checking the range of build years
by = df['build_year'].dropna()
print(f"Build year range: {int(by.min())} to {int(by.max())}")
print(f"After 2026: {(by > 2026).sum()} properties")

Build year range: 1500 to 2028
After 2026: 96 properties


### 5. price extremes
Couple properties under 10K which might be parking spots or garages listed as properties.
And a few over 5M which could be legit mansions or just weird listings.

In [42]:
# looking at the price edges
p = df['price'].dropna()
print(f"Price range: {p.min():,.0f}EUR to {p.max():,.0f}EUR")
print(f"Under 10K: {(p < 10000).sum()}")
print(f"Over 5M: {(p > 5_000_000).sum()}")

Price range: 2,966EUR to 13,000,000EUR
Under 10K: 2
Over 5M: 10


### 6. column names use different styles
Most columns use snake_case like `livable_surface` but a few use capital letters like `Preschool_distance`.
Not a big deal but if it bugs us we can rename them to match.

In [43]:
# finding columns that aren't fully lowercase
for col in df.columns:
    if col != col.lower():
        print(f"  {col} -> could rename to {col.lower()}")

  Preschool_distance -> could rename to preschool_distance
  Train_station_distance -> could rename to train_station_distance
  Supermarket_distance -> could rename to supermarket_distance


## Quick Stats
Handy numbers to have ready when talking with the team.

In [44]:
# just a summary of where we're at
print(f"Total properties: {len(df)}")
print(f"Houses: {(df['property_type'] == 'house').sum()}")
print(f"Apartments: {(df['property_type'] == 'apartment').sum()}")
print(f"Provinces covered: {df['province'].nunique()}")
print(f"Median price: {df['price'].median():,.0f}EUR")
print(f"Mean price: {df['price'].mean():,.0f}EUR")
print()
print("Properties per province:")
print(df['province'].value_counts().to_string())

Total properties: 16729
Houses: 10859
Apartments: 5870
Provinces covered: 11
Median price: 327,000EUR
Mean price: 417,104EUR

Properties per province:
province
vlaams-brabant    2149
antwerp           2077
hainaut           2027
liege             1927
brabant-wallon    1497
brussels          1470
west-flanders     1191
east-flanders     1187
limburg           1133
namur             1063
luxembourg        1008
